# Bronze Layer - Raw Data Ingestion
This notebook reads source data from the Unity Catalog volume and ingests all sheets into the `dev_bronze` catalog as raw tables.

## Step 0: Install Required Libraries

In [0]:
%pip install xlrd openpyxl --quiet

## Step 1: Create the Bronze Catalog and Schema

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS dev_bronze MANAGED LOCATION 's3://databricks-workspace-stack-9a61f-bucket/unity-catalog/6483314808213883/zyad_demo'")
spark.sql("USE CATALOG dev_bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS raw")
spark.sql("USE SCHEMA raw")

## Step 2: Discover Source Files and Import Libraries

In [0]:
import os
import pandas as pd
from pyspark.sql import DataFrame

source_path = "/Volumes/catalog_demo/dev/source_dataset"

files = os.listdir(source_path)
print(f"Files found in source volume: {files}")

## Step 3: Define Ingestion Functions

In [0]:
def clean_table_name(name: str) -> str:
    return name.strip().lower().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "")

def ingest_excel_file(file_path: str):
    xl = pd.ExcelFile(file_path)
    sheet_names = xl.sheet_names
    print(f"Found {len(sheet_names)} sheets in {os.path.basename(file_path)}: {sheet_names}")

    for sheet_name in sheet_names:
        print(f"\nProcessing sheet: '{sheet_name}'")
        pdf = xl.parse(sheet_name)

        if pdf.empty:
            print(f"  Skipping empty sheet: '{sheet_name}'")
            continue

        pdf.columns = [clean_table_name(str(col)) for col in pdf.columns]

        for col in pdf.columns:
            if pdf[col].dtype == 'object':
                pdf[col] = pdf[col].astype(str)
        pdf = pdf.replace('nan', None)

        spark_df = spark.createDataFrame(pdf)

        table_name = clean_table_name(sheet_name)
        full_table_name = f"dev_bronze.raw.{table_name}"

        spark_df.write.mode("overwrite").saveAsTable(full_table_name)
        print(f"  Written table: {full_table_name} ({spark_df.count()} rows, {len(spark_df.columns)} columns)")

def ingest_csv_file(file_path: str):
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    table_name = clean_table_name(file_name)
    full_table_name = f"dev_bronze.raw.{table_name}"

    spark_df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)
    spark_df.write.mode("overwrite").saveAsTable(full_table_name)
    print(f"Written table: {full_table_name} ({spark_df.count()} rows, {len(spark_df.columns)} columns)")

## Step 4: Process All Files

In [0]:
for file_name in files:
    file_path = os.path.join(source_path, file_name)
    print(f"\n{'='*60}")
    print(f"Processing file: {file_name}")
    print(f"{'='*60}")

    if file_name.endswith((".xlsx", ".xls")):
        ingest_excel_file(file_path)
    elif file_name.endswith(".csv"):
        ingest_csv_file(file_path)
    else:
        print(f"  Skipping unsupported file type: {file_name}")

## Step 5: Verify Ingested Tables

In [0]:
tables = spark.sql("SHOW TABLES IN dev_bronze.raw").collect()
print(f"\nTotal tables created in dev_bronze.raw: {len(tables)}\n")
for t in tables:
    row_count = spark.sql(f"SELECT COUNT(*) as cnt FROM dev_bronze.raw.{t.tableName}").collect()[0].cnt
    print(f"  - {t.tableName}: {row_count} rows")